# Fábrica dos Cortes

Cortes verticais para TikTok, Reels e Shorts + compilado **Melhores Momentos** para YouTube.

**Como usar:**
1. Rode a célula **1. Instalar** e espere terminar.
2. Rode a célula **2. Iniciar**.
3. Abra o link `.gradio.live` que aparecer e cole sua chave Gemini na tela.

> Se o download do YouTube falhar com "not a bot", pare a célula 2, rode a célula **3. Cookies** e inicie de novo.


# Suporte

**WhatsApp direto:** [+55 18 98122-3199](https://wa.me/5518981223199)

**Grupo de alunos:** [entrar no grupo](https://chat.whatsapp.com/G0yGg1Rl0K3J8OUEChfc4s)

Site oficial: https://fabricadecortes.com


# O que esta versao entrega
- Cortes verticais 9:16 com legendas e enquadramento no rosto.
- Melhores Momentos 16:9 com capítulos, SRT e descrição para YouTube.
- Marca d'água opcional e encerramento de 11,8 s no fim do vídeo.
- Prompt próprio para cortes curtos e para compilados longos.
- Resgate automático quando a IA entrega menos momentos do que o pedido.
- Quantidade exata de cortes: se a IA entregar menos, o app pede os que faltam.


In [ ]:
#@title 🛠️ 1. Instalar (leva de 5 a 10 minutos)
import os
import shutil
import subprocess
import sys

REPO = "https://github.com/bazotepedro-oss/forjaviral.git"
ROOT = "/content/forjaviral"
PY = os.path.join(ROOT, ".venv", "bin", "python")


def rodar(comando, etapa, obrigatorio=True):
    print(f"\n>> {etapa}")
    resultado = subprocess.run(comando, shell=True, text=True, capture_output=True)
    if resultado.stdout:
        print(resultado.stdout[-3000:])
    if resultado.returncode != 0:
        print(resultado.stderr[-4000:])
        if obrigatorio:
            raise SystemExit(f"FALHOU: {etapa}. Copie as ultimas linhas acima.")
        print(f"[AVISO] {etapa} falhou, seguindo em frente.")
        return False
    return True


print("1/5  Limpando instalacao anterior...")
os.chdir("/content")
if os.path.exists(ROOT):
    shutil.rmtree(ROOT)

print("2/5  Baixando o codigo mais recente do GitHub...")
rodar(f"git clone --depth 1 {REPO} {ROOT}", "git clone")
os.chdir(ROOT)

print("3/5  Dependencias do sistema...")
# libcudnn8 saiu da imagem do Colab. O PyTorch do pip ja traz o CUDA/CuDNN.
rodar(
    "apt-get update -qq && apt-get install -y -qq ffmpeg xvfb libgl1 libglib2.0-0",
    "apt-get",
    obrigatorio=False,
)
if shutil.which("ffmpeg") is None:
    raise SystemExit("ffmpeg nao encontrado. Rode a celula 3 de novo ou reinicie o ambiente.")

print("4/5  Python 3.11 e ambiente virtual...")
rodar(f"{sys.executable} -m pip install -q uv", "instalar uv")
rodar("uv python install 3.11", "baixar Python 3.11")
rodar("uv venv --python 3.11 .venv", "criar ambiente virtual")

print("5/5  Bibliotecas de IA (pode demorar)...")
rodar("uv pip install --python .venv git+https://github.com/m-bain/whisperx.git", "whisperx")
rodar("uv pip install --python .venv -r requirements.txt", "requirements.txt")
rodar("uv pip install --python .venv yt-dlp pytubefix", "yt-dlp e pytubefix")

open("_check_env.py", "w", encoding="utf-8").write(
    "import gradio, torch, yt_dlp, pytubefix\n"
    "print('gradio', gradio.__version__)\n"
    "print('torch', torch.__version__)\n"
    "print('CUDA', torch.cuda.is_available())\n"
)
rodar(f"{PY} _check_env.py", "checagem final")

os.makedirs(os.path.join(ROOT, "VIRALS"), exist_ok=True)
print("\nInstalacao concluida. Agora rode a celula 2.")


In [ ]:
#@title 🚀 2. Iniciar a Fabrica dos Cortes
import os
import re
import subprocess
import sys

ROOT = "/content/forjaviral"
PY = os.path.join(ROOT, ".venv", "bin", "python")

os.chdir(ROOT)
os.environ["DISPLAY"] = ":1.0"
os.environ["MPLBACKEND"] = "Agg"
os.environ["PYTHONUNBUFFERED"] = "1"
os.makedirs("VIRALS", exist_ok=True)
os.makedirs("logs", exist_ok=True)

# Tela virtual: algumas bibliotecas de video exigem um display.
subprocess.Popen(
    "Xvfb :1 -screen 0 2560x1440x8",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("Abrindo a interface. O link aparece destacado abaixo.\n")

log_path = os.path.join("logs", "colab_app.log")
processo = subprocess.Popen(
    [PY, "webui/app.py", "--colab"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    errors="replace",
)

link = None
with open(log_path, "w", encoding="utf-8", errors="replace") as log:
    for linha in processo.stdout:
        sys.stdout.write(linha)
        sys.stdout.flush()
        log.write(linha)
        if link is None:
            achado = re.search(r"https://[a-zA-Z0-9.-]+\.gradio\.live", linha)
            if achado:
                link = achado.group(0)
                print("\n" + "=" * 64)
                print("LINK DA FABRICA DOS CORTES:")
                print(link)
                print("=" * 64 + "\n")

processo.wait()

print("\nO aplicativo encerrou com codigo:", processo.returncode)
if link:
    print("Link desta execucao:", link)
else:
    print("Nenhum link .gradio.live apareceu.")
    print("As ultimas linhas abaixo mostram o motivo (share bloqueado ou erro):")
    with open(log_path, encoding="utf-8", errors="replace") as log:
        print("\n".join(log.read().splitlines()[-25:]))


In [ ]:
#@title 🍪 3. Opcional: cookies do YouTube (use se o download falhar)
import os
import shutil

try:
    from google.colab import files
except ImportError:
    files = None

DESTINO = "/content/forjaviral/cookies.txt"

print("O YouTube bloqueia downloads vindos de servidores (IP do Colab).")
print("Se aparecer 'Sign in to confirm you are not a bot', use esta celula.")
print()
print("Como obter o cookies.txt:")
print("1. Instale a extensao 'Get cookies.txt LOCALLY' no Chrome ou Edge.")
print("2. Abra o YouTube logado na sua conta.")
print("3. Exporte o arquivo cookies.txt.")
print("4. Envie o arquivo abaixo.")
print()

if files is None:
    print("Esta celula so funciona no Colab.")
else:
    enviados = files.upload()
    salvo = False
    for nome in enviados:
        if not nome.lower().endswith(".txt"):
            continue
        shutil.copyfile(nome, DESTINO)
        shutil.copyfile(nome, "/content/cookies.txt")
        tamanho = os.path.getsize(DESTINO)
        print(f"Salvo em: {DESTINO} ({tamanho} bytes)")
        with open(DESTINO, "r", encoding="utf-8", errors="ignore") as fh:
            primeira = fh.readline().strip()
        if "netscape" in primeira.lower() or "\t" in primeira:
            print("Formato reconhecido: cookies Netscape.")
        else:
            print("[AVISO] O arquivo nao parece um cookies.txt do Netscape.")
            print("Exporte novamente usando a extensao indicada.")
        salvo = True

    if salvo:
        print("\nAgora rode a celula 2 de novo.")
        print("No log deve aparecer: [YouTube] Usando cookies de: ...cookies.txt")
    else:
        print("\nNenhum arquivo .txt enviado. Nada foi alterado.")


# Créditos

**Fábrica dos Cortes** — projeto independente.

Site: https://fabricadecortes.com  
Suporte: [+55 18 98122-3199](https://wa.me/5518981223199)


`1.3.24`

Alternativa brasileira ao Opus Clip e ao Vidyo.ai, com créditos no Pix.
